<a href="https://colab.research.google.com/github/vinodhr91/personal/blob/main/investor_copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pypdf sentence-transformers qdrant-client beautifulsoup4 requests google-genai
print("✅ Libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 13.7 MB/s eta 0:00:00
✅ Libraries installed!


In [ ]:
import os
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

PDF_PATH = "The Five Rules for Successful Stock Investing (1).pdf"

# Fallback check in case the file was uploaded without the .pdf extension
if not os.path.exists(PDF_PATH):
    if os.path.exists("The Five Rules for Successful Stock Investing (1)"):
        PDF_PATH = "The Five Rules for Successful Stock Investing (1)"
    else:
        raise FileNotFoundError(f"Please upload '{PDF_PATH}' into Colab's left sidebar.")

print(f"📖 Reading {PDF_PATH}...")
reader = PdfReader(PDF_PATH)

pages_data = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text and text.strip():
        pages_data.append({"page": i + 1, "text": text})

# Simple chunking function
def chunk_text(text, chunk_size=600, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

all_chunks = []
for item in pages_data:
    for chunk in chunk_text(item["text"]):
        if len(chunk.strip()) > 50:
            all_chunks.append({"page": item["page"], "text": chunk})

print(f"Total chunks created: {len(all_chunks)}")
print("Loading embedding model (BAAI/bge-small-en-v1.5)...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Initialize Qdrant local storage
qdrant_client = QdrantClient(path="./qdrant_dorsey_db")
collection_name = "pat_dorsey_rules"

if qdrant_client.collection_exists(collection_name):
    qdrant_client.delete_collection(collection_name)

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

# Embed and upload in batches
points = []
for idx, chunk_info in enumerate(all_chunks):
    vector = embed_model.encode(chunk_info["text"]).tolist()
    points.append(
        PointStruct(
            id=idx,
            vector=vector,
            payload={"page": chunk_info["page"], "text": chunk_info["text"]}
        )
    )

batch_size = 100
for i in range(0, len(points), batch_size):
    qdrant_client.upsert(collection_name=collection_name, points=points[i:i + batch_size])

print("✅ Pat Dorsey rules successfully indexed into Qdrant!")

FileNotFoundError: Please upload 'The Five Rules for Successful Stock Investing (1).pdf' into Colab's left sidebar.

In [ ]:
import os
print("Files in current directory:")
for f in os.listdir("."):
    if f.endswith(".pdf") or "dorsey" in f.lower() or "rules" in f.lower():
        print("  ->", repr(f))

Files in current directory:


In [ ]:
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"✅ Successfully uploaded: {filename}")

Saving The Five Rules for Successful Stock Investing (1).pdf to The Five Rules for Successful Stock Investing (1).pdf
✅ Successfully uploaded: The Five Rules for Successful Stock Investing (1).pdf


In [ ]:
import os

# Grab the uploaded file name automatically
uploaded_pdfs = [f for f in os.listdir(".") if f.lower().endswith(".pdf")]

if not uploaded_pdfs:
    raise FileNotFoundError("Still no PDF detected. Please run the cell above to upload.")

PDF_PATH = uploaded_pdfs[0]
print(f"🎯 Using PDF file: {PDF_PATH}")

🎯 Using PDF file: The Five Rules for Successful Stock Investing (1).pdf


In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print(f"📖 Reading {PDF_PATH}...")
reader = PdfReader(PDF_PATH)

pages_data = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text and text.strip():
        pages_data.append({"page": i + 1, "text": text})

def chunk_text(text, chunk_size=600, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

all_chunks = []
for item in pages_data:
    for chunk in chunk_text(item["text"]):
        if len(chunk.strip()) > 50:
            all_chunks.append({"page": item["page"], "text": chunk})

print(f"Total chunks created: {len(all_chunks)}")
print("Loading embedding model (BAAI/bge-small-en-v1.5)...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Setup local Qdrant storage
qdrant_client = QdrantClient(path="./qdrant_dorsey_db")
collection_name = "pat_dorsey_rules"

if qdrant_client.collection_exists(collection_name):
    qdrant_client.delete_collection(collection_name)

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

print("Embedding chunks and indexing into Qdrant...")
points = []
for idx, chunk_info in enumerate(all_chunks):
    vector = embed_model.encode(chunk_info["text"]).tolist()
    points.append(
        PointStruct(
            id=idx,
            vector=vector,
            payload={"page": chunk_info["page"], "text": chunk_info["text"]}
        )
    )

# Batch upsert
batch_size = 100
for i in range(0, len(points), batch_size):
    qdrant_client.upsert(collection_name=collection_name, points=points[i:i + batch_size])

print("✅ Pat Dorsey rules successfully indexed into Qdrant!")

📖 Reading The Five Rules for Successful Stock Investing (1).pdf...
Total chunks created: 1619
Loading embedding model (BAAI/bge-small-en-v1.5)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding chunks and indexing into Qdrant...
✅ Pat Dorsey rules successfully indexed into Qdrant!


In [ ]:
import requests
from bs4 import BeautifulSoup
from google import genai
from google.colab import userdata

# Initialize Gemini Client separately to avoid naming conflicts
ai_client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

def query_dorsey_rules(query_text, top_k=4):
    query_vector = embed_model.encode(query_text).tolist()
    response = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k
    )
    return "\n---\n".join([hit.payload["text"] for hit in response.points])

def get_screener_data(symbol: str) -> dict:
    clean_sym = symbol.strip().upper()
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    url = f"https://www.screener.in/company/{clean_sym}/consolidated/"
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        url = f"https://www.screener.in/company/{clean_sym}/"
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            return {"error": f"Failed to fetch data for ticker '{clean_sym}' from Screener.in"}

    soup = BeautifulSoup(response.content, "html.parser")
    company_name = soup.find("h1").text.strip() if soup.find("h1") else clean_sym

    ratios = {}
    top_ratios_ul = soup.find("ul", id="top-ratios")
    if top_ratios_ul:
        for li in top_ratios_ul.find_all("li"):
            name_el = li.find("span", class_="name")
            val_el = li.find("span", class_="number")
            if name_el and val_el:
                ratios[name_el.get_text(strip=True)] = val_el.get_text(strip=True)

    about_div = soup.find("div", class_="about")
    about_text = about_div.get_text(" ", strip=True) if about_div else "N/A"

    return {
        "company_name": company_name,
        "symbol": clean_sym,
        "ratios": ratios,
        "about": about_text
    }

def analyze_stock_with_dorsey(symbol: str, model_id: str = "gemini-2.5-flash"):
    print(f"\n🔍 Fetching Screener metrics for {symbol.upper()}...")
    screener_data = get_screener_data(symbol)

    if "error" in screener_data:
        print(f"❌ Error: {screener_data['error']}")
        return

    metrics_str = "\n".join([f"- {k}: {v}" for k, v in screener_data["ratios"].items()])

    retrieval_query = (
        "Economic moats, return on capital ROE ROCE, debt leverage, "
        "cash flow generation, competitive advantage, margin of safety"
    )
    dorsey_excerpts = query_dorsey_rules(retrieval_query, top_k=4)

    prompt = f"""
You are an expert fundamental equity research copilot applying Pat Dorsey's
investment framework from "The Five Rules for Successful Stock Investing".

### COMPANY UNDER ANALYSIS:
- Name: {screener_data['company_name']} ({screener_data['symbol']})
- About: {screener_data['about']}

### LIVE FINANCIAL METRICS (Screener.in):
{metrics_str}

### RETRIEVED DORSEY FRAMEWORK PRINCIPLES:
{dorsey_excerpts}

---
### OBJECTIVE:
Synthesize an investment memo evaluating {screener_data['company_name']} against Dorsey's 5 core rules:
1. **Do Your Homework (Business Model & Understanding):** What does this business actually sell?
2. **Find Economic Moats:** Evaluate its competitive moat (Intangible Assets, Switching Costs, Cost Advantages, Network Effects). Check if ROCE/ROE confirms excess economic returns.
3. **Have a Margin of Safety & Valuation Discipline:** Assess current valuation multiples (P/E, Market Cap, Book Value, etc.).
4. **Hold for the Long Haul:** Is this a durable compounding machine or a cyclical trap?
5. **Know When to Sell:** Red flags, leverage risk (Debt-to-Equity), or margin erosion.

Conclude with a clear **Verdict Summary**:
- **Moat Rating:** (None / Narrow / Wide)
- **Financial Health:** (Fragile / Moderate / Robust)
- **Actionable Takeaway:** Grounded synthesis for a long-term investor.
"""

    print("🧠 Analyzing metrics against Pat Dorsey's framework...")
    response = ai_client.models.generate_content(
        model=model_id,
        contents=prompt
    )

    print("\n" + "="*80)
    print(f"PAT DORSEY INVESTMENT MEMO: {screener_data['company_name']}")
    print("="*80 + "\n")
    print(response.text)

# Run it on TITAN
analyze_stock_with_dorsey("TITAN")


🔍 Fetching Screener metrics for TITAN...
🧠 Analyzing metrics against Pat Dorsey's framework...


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:
def analyze_stock_with_dorsey(symbol: str, model_id: str = "gemini-3.8-flash"):
    print(f"\n🔍 Fetching Screener metrics for {symbol.upper()}...")
    screener_data = get_screener_data(symbol)

    if "error" in screener_data:
        print(f"❌ Error: {screener_data['error']}")
        return

    metrics_str = "\n".join([f"- {k}: {v}" for k, v in screener_data["ratios"].items()])

    retrieval_query = (
        "Economic moats, return on capital ROE ROCE, debt leverage, "
        "cash flow generation, competitive advantage, margin of safety"
    )
    dorsey_excerpts = query_dorsey_rules(retrieval_query, top_k=4)

    prompt = f"""
You are an expert fundamental equity research copilot applying Pat Dorsey's
investment framework from "The Five Rules for Successful Stock Investing".

### COMPANY UNDER ANALYSIS:
- Name: {screener_data['company_name']} ({screener_data['symbol']})
- About: {screener_data['about']}

### LIVE FINANCIAL METRICS (Screener.in):
{metrics_str}

### RETRIEVED DORSEY FRAMEWORK PRINCIPLES:
{dorsey_excerpts}

---
### OBJECTIVE:
Synthesize an investment memo evaluating {screener_data['company_name']} against Dorsey's 5 core rules:
1. **Do Your Homework (Business Model & Understanding):** What does this business actually sell?
2. **Find Economic Moats:** Evaluate its competitive moat (Intangible Assets, Switching Costs, Cost Advantages, Network Effects). Check if ROCE/ROE confirms excess economic returns.
3. **Have a Margin of Safety & Valuation Discipline:** Assess current valuation multiples (P/E, Market Cap, Book Value, etc.).
4. **Hold for the Long Haul:** Is this a durable compounding machine or a cyclical trap?
5. **Know When to Sell:** Red flags, leverage risk (Debt-to-Equity), or margin erosion.

Conclude with a clear **Verdict Summary**:
- **Moat Rating:** (None / Narrow / Wide)
- **Financial Health:** (Fragile / Moderate / Robust)
- **Actionable Takeaway:** Grounded synthesis for a long-term investor.
"""

    print("🧠 Analyzing metrics against Pat Dorsey's framework...")
    response = ai_client.models.generate_content(
        model=model_id,
        contents=prompt
    )

    print("\n" + "="*80)
    print(f"PAT DORSEY INVESTMENT MEMO: {screener_data['company_name']}")
    print("="*80 + "\n")
    print(response.text)

# Run it
analyze_stock_with_dorsey("TITAN")


🔍 Fetching Screener metrics for TITAN...
🧠 Analyzing metrics against Pat Dorsey's framework...

PAT DORSEY INVESTMENT MEMO: Titan Company Ltd

### Investment Memo: Titan Company Ltd (TITAN)
**Framework:** Pat Dorsey’s *Five Rules for Successful Stock Investing*

---

### 1. Rule 1: Do Your Homework (Business Model & Economics)
*Dorsey Principle: Understand how the business generates revenue, where its cash comes from, and what structural tailwinds support it.*

* **What Titan Actually Sells:** While perceived by many as a watchmaker, Titan is predominantly an **organized luxury and jewellery retailer**. 
  * **Jewellery (~85–90% of revenue):** Anchored by flagship brand **Tanishq**, along with **Mia** (workwear jewellery), **Zoya** (high-end luxury), and **CaratLane** (omnichannel/digital-first).
  * **Watches & Wearables (~7–9% of revenue):** Titan, Fastrack, Sonata, and smart wearables.
  * **Emerging Segments (~2–4% of revenue):** Eyewear (Titan Eye+), Fragrances (Skinn), and India

In [ ]:
import ipywidgets as widgets
from IPython.display import display

ticker_input = widgets.Text(
    value='ITC',
    placeholder='Enter NSE/BSE Symbol (e.g. TCS, RELIANCE, HDFCBANK)',
    description='Ticker:',
    disabled=False
)

run_button = widgets.Button(
    description='Generate Memo',
    button_style='success',
    tooltip='Analyze stock using Pat Dorsey RAG',
    icon='search'
)

output_area = widgets.Output()

def on_button_clicked(b):
    output_area.clear_output()
    with output_area:
        ticker = ticker_input.value.strip().upper()
        if ticker:
            analyze_stock_with_dorsey(ticker)
        else:
            print("Please enter a valid ticker symbol.")

run_button.on_click(on_button_clicked)

display(widgets.HBox([ticker_input, run_button]), output_area)

Output()